# Notebook 06: Choosing an Embedding Model for Your RAG Use Case

There is no universally best embedding model: the best choice depends on your corpus, query style, language, latency, and hosting constraints. This notebook compares candidates on the same labeled query–passage benchmark using retrieval quality, query encoding time, and vector dimensions.

The included benchmark is a runnable demonstration only. Replace it with representative, reviewed queries and relevant chunk IDs from your application before choosing a model. A dozen synthetic examples cannot establish which model is best for your use case.

### Learning objectives
- Build a labeled retrieval benchmark from real questions and relevant chunks.
- Compare Recall@k, MRR@k, and nDCG@k.
- Apply each model's documented query and passage formatting.
- Consider latency and vector size alongside relevance.
- Understand why a retrieval benchmark does not measure final answer quality.

## 1. Install dependencies

Model weights download when first loaded. For production comparisons, pin package and model revisions and record the hardware used.

In [ ]:
%pip install -q sentence-transformers pandas

## 2. Prepare a labeled benchmark

Use the same chunks and metadata filters as your application. Include paraphrases, acronyms, exact identifiers, multilingual or domain-specific queries where relevant, and unanswerable questions. Keep a separate held-out set for final evaluation.

In [ ]:
# Synthetic demo corpus. Replace with representative chunks from your own knowledge base.
corpus = {
    "wifi-1": "On macOS, open System Settings, choose Wi-Fi, and toggle Wi-Fi off and on.",
    "wifi-2": "If Wi-Fi still fails, restart the router and forget then rejoin the network.",
    "refund-1": "You can request a refund within 30 days of purchase by contacting support with your order ID.",
    "refund-2": "Refunds return to the original payment method and may take 5 to 10 business days.",
    "security-1": "Enable two-factor authentication from Account Settings under Security.",
    "security-2": "Use a unique password and store it in a trusted password manager.",
    "billing-1": "Download invoices from the Billing section of your account dashboard.",
    "billing-2": "To change the card for future payments, update the default payment method in Billing.",
    "shipping-1": "Standard shipping usually takes 3 to 5 business days after dispatch.",
    "shipping-2": "Use the tracking link in your dispatch email to follow a package.",
    "account-1": "To change your account email, open Profile Settings and verify the new address.",
    "account-2": "If you cannot sign in, use Forgot password on the sign-in page."
}

# Each query has one or more known relevant chunk IDs.
queries = [
    {"query": "How do I turn Wi-Fi back on on my Mac?", "relevant_ids": ["wifi-1"]},
    {"query": "My laptop has no internet. What should I try?", "relevant_ids": ["wifi-2", "wifi-1"]},
    {"query": "Can I get my money back after I buy something?", "relevant_ids": ["refund-1"]},
    {"query": "How long until a refund appears on my card?", "relevant_ids": ["refund-2"]},
    {"query": "Where do I switch on two step login?", "relevant_ids": ["security-1"]},
    {"query": "How can I make my password safer?", "relevant_ids": ["security-2"]},
    {"query": "I need a copy of my receipt for expenses.", "relevant_ids": ["billing-1"]},
    {"query": "Where can I replace my saved credit card?", "relevant_ids": ["billing-2"]},
    {"query": "When should my order arrive?", "relevant_ids": ["shipping-1"]},
    {"query": "Can I see where my parcel is now?", "relevant_ids": ["shipping-2"]},
    {"query": "How do I update the email address on my account?", "relevant_ids": ["account-1"]},
    {"query": "I forgot my login password.", "relevant_ids": ["account-2"]},
]
assert all(set(q["relevant_ids"]).issubset(corpus) for q in queries)
print(f"{len(corpus)} passages, {len(queries)} labeled queries")

## 3. Choose candidate models

These are examples to demonstrate the comparison, not a ranking or recommendation. Select candidates that fit your language, privacy, deployment, licensing, and hardware needs. Check each model's current model card and license before use.

Retrieval encoders may require asymmetric query/passage prefixes. Configure them exactly as specified in their documentation. Include your current production model as a baseline.

In [ ]:
candidates = [
    {
        "name": "all-MiniLM-L6-v2",
        "model_id": "sentence-transformers/all-MiniLM-L6-v2",
        "query_prefix": "", "passage_prefix": "",
    },
    {
        "name": "multilingual-e5-small",
        "model_id": "intfloat/multilingual-e5-small",
        "query_prefix": "query: ", "passage_prefix": "passage: ",
    },
    {
        "name": "bge-small-en-v1.5",
        "model_id": "BAAI/bge-small-en-v1.5",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "passage_prefix": "",
    },
]
# Add/remove candidates and configure formatting per the model's documentation.

## 4. Define retrieval metrics

- **Recall@k**: share of known relevant passages found in the top k.
- **MRR@k**: rewards placing the first relevant passage near the top.
- **nDCG@k**: rewards relevant passages near the top; can also handle graded judgments.

Set k to the number of chunks your pipeline passes downstream. Review per-query results as averages can hide failures on important query types.

In [ ]:
import math
import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

K = 5  # Match the retrieval depth in your RAG pipeline.

def recall_at_k(ranked_ids, relevant_ids, k):
    relevant = set(relevant_ids)
    return len(set(ranked_ids[:k]) & relevant) / len(relevant) if relevant else 0.0

def reciprocal_rank_at_k(ranked_ids, relevant_ids, k):
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(ranked_ids[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(ranked_ids, relevant_ids, k):
    # Binary relevance; use graded gains if you have graded judgments.
    relevant = set(relevant_ids)
    gains = [1.0 if doc_id in relevant else 0.0 for doc_id in ranked_ids[:k]]
    dcg = sum(gain / math.log2(rank + 1) for rank, gain in enumerate(gains, 1))
    ideal = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal + 1))
    return dcg / idcg if idcg else 0.0

## 5. Encode and evaluate

This evaluator uses exact cosine search across all passages, suitable for a small benchmark. For a large collection, also benchmark the actual vector index and its approximate-search settings.

Encoding time here is a rough measurement after model loading. For useful latency figures, warm up, repeat runs, report percentiles, use realistic batches, and test on deployment hardware. Normalized embeddings make dot product equal cosine similarity.

In [ ]:
passage_ids = list(corpus)
passage_texts = [corpus[doc_id] for doc_id in passage_ids]

def evaluate_candidate(candidate):
    model = SentenceTransformer(candidate["model_id"])
    passage_vectors = model.encode(
        [candidate["passage_prefix"] + text for text in passage_texts],
        normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False
    )
    formatted_queries = [candidate["query_prefix"] + q["query"] for q in queries]
    start = time.perf_counter()
    query_vectors = model.encode(
        formatted_queries, normalize_embeddings=True, convert_to_numpy=True,
        show_progress_bar=False
    )
    elapsed = time.perf_counter() - start

    rows = []
    for item, vector in zip(queries, query_vectors):
        order = np.argsort(-(passage_vectors @ vector))
        ranked = [passage_ids[i] for i in order]
        relevant = item["relevant_ids"]
        rows.append({
            "query": item["query"],
            "recall_at_k": recall_at_k(ranked, relevant, K),
            "mrr_at_k": reciprocal_rank_at_k(ranked, relevant, K),
            "ndcg_at_k": ndcg_at_k(ranked, relevant, K),
            "top_ids": ranked[:K],
        })
    return {
        "name": candidate["name"],
        "model_id": candidate["model_id"],
        "dimensions": int(passage_vectors.shape[1]),
        "query_encode_ms_per_query": 1000 * elapsed / len(queries),
        "metrics": pd.DataFrame(rows),
    }

results = []
for candidate in candidates:
    print(f"Evaluating {candidate['name']} ...")
    results.append(evaluate_candidate(candidate))

summary = pd.DataFrame([
    {
        "model": r["name"],
        "Recall@k": r["metrics"]["recall_at_k"].mean(),
        "MRR@k": r["metrics"]["mrr_at_k"].mean(),
        "nDCG@k": r["metrics"]["ndcg_at_k"].mean(),
        "dimensions": r["dimensions"],
        "query encode ms/query": r["query_encode_ms_per_query"],
    }
    for r in results
]).sort_values(["Recall@k", "MRR@k"], ascending=False).reset_index(drop=True)
summary

## 6. Inspect failures

A good average can hide misses on an important slice of your workload. Inspect the worst cases and group them by language, product area, query length, exact identifiers, or other categories that matter to your application.

In [ ]:
per_query = pd.DataFrame([
    {"model": r["name"], **row}
    for r in results
    for row in r["metrics"].to_dict(orient="records")
])
per_query.sort_values(
    ["recall_at_k", "mrr_at_k", "model"], ascending=[True, True, True]
).head(20)

## 7. Decide for your application

1. Set requirements first: a minimum retrieval score, latency target, memory/storage budget, language coverage, and license/privacy constraints.
2. Compare on representative, reviewed, held-out queries. The included demo is only a code example.
3. Weigh quality against cost. Vector storage is roughly `number_of_chunks × dimensions × bytes_per_value`, before index overhead and metadata.
4. Inspect misses. Chunking, metadata filters, hybrid BM25+dense search, or reranking may help more than changing the encoder.
5. Validate end to end: answer correctness, citation quality, retrieval latency, and cost. Better retrieval metrics do not guarantee better generated answers.
6. Record dataset version, chunking, model ID/revision, prefixes, metrics, hardware, and results so the experiment is reproducible.

Choose the model that meets your quality floor and operational constraints, then validate it in your complete RAG application.

## Further experiments

After narrowing candidates, change one factor at a time: chunk size and overlap, query instructions, filters, hybrid search, or a cross-encoder reranker. Keep a separate final test set for the model you plan to ship. Consult official model documentation for language coverage, input limits, formatting, normalization, and license terms.